<a href="https://colab.research.google.com/github/Baranetharan-ds/Datascience-bootcamp/blob/main/Logistic_Regression_Social_Network_Ads_with_finetuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Logistic Regression - Social Network Ads

- Classification algorithm (predicts a category, not a number)
- Binary Classification: output is 0 or 1 (e.g., Will the user buy? Yes / No)
- Uses Sigmoid function to convert output between 0 and 1
- Dataset: Age, Estimated Salary -> Purchased (0 or 1)

## Step 1 - Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

## Step 2 - Load Dataset

In [ ]:
df = pd.read_csv("https://raw.githubusercontent.com/nunnarilabs/ml/master/Social_Network_Ads.csv")

In [ ]:
df

## Step 3 - EDA (Exploratory Data Analysis)

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df.isnull().sum()

In [ ]:
df = df.dropna()
df.isnull().sum()

In [ ]:
# How many users purchased vs did not purchase
df['Purchased'].value_counts()

In [ ]:
# Bar chart - class distribution (0 = Not Purchased, 1 = Purchased)
sns.countplot(x='Purchased', data=df)
plt.title('Purchased vs Not Purchased')
plt.xlabel('Purchased (0 = No, 1 = Yes)')
plt.ylabel('Count')
plt.show()

In [ ]:
# Distribution of Age
sns.histplot(df['Age'], bins=20, kde=True)
plt.title('Age Distribution')
plt.xlabel('Age')
plt.show()

In [ ]:
# Distribution of EstimatedSalary
sns.histplot(df['EstimatedSalary'], bins=20, kde=True, color='orange')
plt.title('Estimated Salary Distribution')
plt.xlabel('EstimatedSalary')
plt.show()

In [ ]:
# Scatter plot - Age vs Salary, colored by Purchased
# This shows whether purchased users cluster in a specific region
sns.scatterplot(x='Age', y='EstimatedSalary', hue='Purchased', data=df, palette='Set1')
plt.title('Age vs Salary (colored by Purchased)')
plt.show()

In [ ]:
# Box plot - Age split by Purchased
# Shows if older users tend to purchase more
sns.boxplot(x='Purchased', y='Age', data=df)
plt.title('Age vs Purchased')
plt.show()

In [ ]:
# Box plot - Salary split by Purchased
sns.boxplot(x='Purchased', y='EstimatedSalary', data=df)
plt.title('Estimated Salary vs Purchased')
plt.show()

## Step 4 - Finding Correlation

In [ ]:
# Correlation values between all numeric columns
df.corr(numeric_only=True)

## Step 5 - Feature Selection using Heatmap

In [ ]:
sns.heatmap(df.corr(numeric_only=True), annot=True, cmap='coolwarm')
plt.title('Correlation Heatmap')
plt.show()

In [ ]:
# Age and EstimatedSalary have good correlation with Purchased
X = df[['Age', 'EstimatedSalary']]
Y = df['Purchased']

## Step 6 - Feature Scaling

- Age is in range 18-60, Salary is in range 15000-150000
- StandardScaler brings all values to the same scale (mean=0, std=1)

In [ ]:
sc = StandardScaler()
X_scaled = sc.fit_transform(X)

## Step 7 - Train Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X_scaled, Y, test_size=0.20, random_state=42)

In [ ]:
print(X_scaled.shape)
print(X_train.shape)
print(X_test.shape)

## Step 8 - Model Building

In [ ]:
model = LogisticRegression()

### Training the Model

In [ ]:
model.fit(X_train, y_train)

In [ ]:
y_train_pred = model.predict(X_train)
print("Training Accuracy:", accuracy_score(y_train, y_train_pred))

### Testing the Model

In [ ]:
y_test_pred = model.predict(X_test)
print("Testing Accuracy:", accuracy_score(y_test, y_test_pred))

### Predict with New Data

In [ ]:
# Predict for a new person: Age=30, EstimatedSalary=87000
new_data = [[30, 87000]]
new_data_scaled = sc.transform(new_data)
prediction = model.predict(new_data_scaled)
print("Prediction (0=Not Purchased, 1=Purchased):", prediction[0])

In [ ]:
probability = model.predict_proba(new_data_scaled)
print("Probability [Not Purchased, Purchased]:", probability)

## Step 9 - Evaluation Metrics

- **Accuracy**: How many predictions were correct overall
- **Confusion Matrix**: Shows True Positives, True Negatives, False Positives, False Negatives
- **Precision**: Out of all predicted Purchased, how many actually purchased
- **Recall**: Out of all actual Purchased, how many did the model catch
- **F1 Score**: Balance between Precision and Recall

In [ ]:
cm = confusion_matrix(y_test, y_test_pred)
print(cm)

In [ ]:
# Confusion Matrix Heatmap
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Not Purchased', 'Purchased'],
            yticklabels=['Not Purchased', 'Purchased'])
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')
plt.show()

In [ ]:
print(classification_report(y_test, y_test_pred))

---
## Step 10 - Improving Model Accuracy

If the model accuracy is around 0.80, we can try the following techniques to improve it.

### Technique 1 - Polynomial Features
- Logistic Regression draws a straight decision boundary
- If the data is not linearly separable, a straight line cannot separate the classes well
- Polynomial Features creates new columns like Age^2, Salary^2, Age x Salary
- This allows the model to learn a curved decision boundary

In [ ]:
# Create polynomial features from the scaled data
poly = PolynomialFeatures(degree=2)
X_poly = poly.fit_transform(X_scaled)

In [ ]:
print("Original features:", X_scaled.shape[1])
print("Polynomial features:", X_poly.shape[1])

In [ ]:
X_train_poly, X_test_poly, y_train_poly, y_test_poly = train_test_split(X_poly, Y, test_size=0.20, random_state=42)

In [ ]:
model_poly = LogisticRegression(max_iter=1000)
model_poly.fit(X_train_poly, y_train_poly)

In [ ]:
y_train_poly_pred = model_poly.predict(X_train_poly)
y_test_poly_pred  = model_poly.predict(X_test_poly)
print("Training Accuracy (Polynomial):", accuracy_score(y_train_poly, y_train_poly_pred))
print("Testing Accuracy  (Polynomial):", accuracy_score(y_test_poly,  y_test_poly_pred))

In [ ]:
# Confusion matrix after polynomial improvement
cm_poly = confusion_matrix(y_test_poly, y_test_poly_pred)
sns.heatmap(cm_poly, annot=True, fmt='d', cmap='Greens',
            xticklabels=['Not Purchased', 'Purchased'],
            yticklabels=['Not Purchased', 'Purchased'])
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix - After Polynomial Features')
plt.show()

### Technique 2 - Compare Accuracy: Before vs After

In [ ]:
# Bar chart comparing accuracy before and after improvement
models    = ['Basic Model', 'Polynomial Model']
train_scores = [
    accuracy_score(y_train, y_train_pred),
    accuracy_score(y_train_poly, y_train_poly_pred)
]
test_scores = [
    accuracy_score(y_test, y_test_pred),
    accuracy_score(y_test_poly, y_test_poly_pred)
]

x     = np.arange(len(models))
width = 0.35

fig, ax = plt.subplots()
ax.bar(x - width/2, train_scores, width, label='Train Accuracy', color='steelblue')
ax.bar(x + width/2, test_scores,  width, label='Test Accuracy',  color='coral')
ax.set_ylabel('Accuracy')
ax.set_title('Model Accuracy: Before vs After')
ax.set_xticks(x)
ax.set_xticklabels(models)
ax.set_ylim(0, 1.1)
ax.legend()
plt.show()

### Technique 3 - Hyperparameter Tuning (C value)

- `C` controls how strict/flexible the decision boundary is
- Low C = simpler model (may underfit)
- High C = complex model (may overfit)
- We try multiple C values and pick the best performing one

In [ ]:
c_values      = [0.01, 0.1, 1, 10, 100]
train_acc_list = []
test_acc_list  = []

for c in c_values:
    m = LogisticRegression(C=c, max_iter=1000)
    m.fit(X_train_poly, y_train_poly)
    train_acc_list.append(accuracy_score(y_train_poly, m.predict(X_train_poly)))
    test_acc_list.append(accuracy_score(y_test_poly,   m.predict(X_test_poly)))

In [ ]:
# Line plot - how C value affects accuracy
plt.plot(c_values, train_acc_list, marker='o', label='Train Accuracy')
plt.plot(c_values, test_acc_list,  marker='s', label='Test Accuracy')
plt.xscale('log')
plt.xlabel('C value (log scale)')
plt.ylabel('Accuracy')
plt.title('Effect of C on Model Accuracy')
plt.legend()
plt.show()

In [ ]:
best_c = c_values[test_acc_list.index(max(test_acc_list))]
print("Best C value   :", best_c)
print("Best Test Accuracy:", max(test_acc_list))